# Step 1.Notebook Purpose

**Purpose of this Model Training Notebook**

##### The goal of this notebook is to:

* Train the selected final model (Logistic Regression) on prepared features, Optimize training configuration,
* Evaluate performance on unseen test data, Persist the trained model and vectorizer for deployment

##### Model selection has already been completed in the previous notebook.

# Step 2.Import Required Libraries

**Purpose :** Import only what is required for training, evaluation, and saving the model.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import joblib


In [2]:
import os
import sys

# Add src folder to Python path
sys.path.append(os.path.abspath("../src"))

# Import custom project functions
from preprocessing import clean_text
from feature_engineering import create_features

print("Custom functions imported successfully.")

Custom functions imported successfully.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Step 3.Load Feature-Engineered Dataset

**Purpose :** Load the dataset created in the feature engineering notebook. This dataset is now model-ready.


In [3]:
feature_path = r"C:\Users\hp\Desktop\Fintech_Complaint_Analysis\data\processed\complaints_features.csv"
df = pd.read_csv(feature_path)
df.head()


,char_length,word_length,stopword_ratio,risk_keyword_flag,product_encoded,issue_encoded,complaint_year,complaint_month,accordance,account,...,xxxx date,xxxx xxxx,xxxx xxxxxxxx,xxxxxxxx,xxxxxxxx balance,xxxxxxxx xxxx,xxxxyear,year,yet,risk_label
0,57,8,0.0,0,6,32,2025,10,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0
1,82,14,0.0,1,6,32,2025,10,0.555462,0.200713,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,1
2,26,5,0.0,0,7,32,2020,5,0.000000,1.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0
3,364,54,0.0,1,6,32,2025,10,0.000000,0.169315,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,1
4,1599,233,0.0,1,6,32,2025,11,0.000000,0.112413,...,0.0,0.409834,0.040575,0.032309,0.0,0.039342,0.0,0.0,0.0,2


# Step 4.Separate Features and Target

**Purpose :** 

Clearly separate: 

X → input features

y → target variable 

In [4]:
df.columns

Index(['char_length', 'word_length', 'stopword_ratio', 'risk_keyword_flag',
       'product_encoded', 'issue_encoded', 'complaint_year', 'complaint_month',
       'accordance', 'account',
       ...
       'xxxx date', 'xxxx xxxx', 'xxxx xxxxxxxx', 'xxxxxxxx',
       'xxxxxxxx balance', 'xxxxxxxx xxxx', 'xxxxyear', 'year', 'yet',
       'risk_label'],
      dtype='object', length=309)

## Create a Risk Score

**Purpose :** Combine multiple signals into a single numeric score.

**We will use:**
    
risk_keyword_flag → already created (0 or 1),and word_length → long complaints often indicate severity

In [5]:
df['risk_score'] = (
    df['risk_keyword_flag'] +
    (df['word_length'] > 100).astype(int)
)


* If complaint has risky keywords → +1
* If complaint is long → +1
* Risk score ∈ {0, 1, 2}

## Convert Risk Score → Target Label

**Purpose :**  Machine learning models need categorical targets, not scores.

In [6]:
def map_risk(score):
    if score == 0:
        return 0   # Low
    elif score == 1:
        return 1   # Medium
    else:
        return 2   # High

df['risk_label'] = df['risk_score'].apply(map_risk)


## Validate the Target Variable

**Purpose :** Ensure the label exists and is balanced.

In [7]:
df['risk_label'].value_counts()

risk_label
2    1448
1    1195
0     681
Name: count, dtype: int64

**This confirms:**  

1.Target created
    
2.Classification problem is valid

In [8]:
df.shape

(3324, 310)

## Check Column Presence 

**Purpose :** Avoid KeyError later.

In [9]:
'risk_label' in df.columns


True

## Remove Helper Column 

**Purpose :** risk_score is only for label creation, not for training.

In [10]:
df.drop(columns=['risk_score'], inplace=True)


## Save Updated Dataset

**Purpose :**  Persist the dataset with features + target.

In [11]:
feature_path = r"C:\Users\hp\Desktop\Fintech_Complaint_Analysis\data\processed\complaints_features.csv"
df.to_csv(feature_path, index=False)

print("Dataset with target variable saved successfully")


Dataset with target variable saved successfully


## Confirm Final Shape

**Purpose :** Final sanity check before model training.

In [12]:
df.shape


(3324, 309)

In [13]:
X = df.drop(columns=['risk_label'])
y = df['risk_label']

In [14]:
X

,char_length,word_length,stopword_ratio,risk_keyword_flag,product_encoded,issue_encoded,complaint_year,complaint_month,accordance,account,...,xxxx balance,xxxx date,xxxx xxxx,xxxx xxxxxxxx,xxxxxxxx,xxxxxxxx balance,xxxxxxxx xxxx,xxxxyear,year,yet
0,57,8,0.0,0,6,32,2025,10,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000
1,82,14,0.0,1,6,32,2025,10,0.555462,0.200713,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000
2,26,5,0.0,0,7,32,2020,5,0.000000,1.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000
3,364,54,0.0,1,6,32,2025,10,0.000000,0.169315,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000
4,1599,233,0.0,1,6,32,2025,11,0.000000,0.112413,...,0.0,0.0,0.409834,0.040575,0.032309,0.0,0.039342,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3319,339,49,0.0,0,6,32,2025,11,0.000000,0.000000,...,0.0,0.0,0.117449,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000
3320,1513,279,0.0,1,10,77,2025,9,0.000000,0.121353,...,0.0,0.0,0.235962,0.000000,0.041854,0.0,0.000000,0.0,0.000000,0.064745
3321,257,39,0.0,1,8,4,2025,9,0.000000,0.253319,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000
3322,737,107,0.0,0,6,32,2025,11,0.245237,0.088615,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000


In [15]:
y

0       0
1       1
2       0
3       1
4       2
       ..
3319    0
3320    2
3321    1
3322    1
3323    1
Name: risk_label, Length: 3324, dtype: int64

In [16]:
print(X.shape)
print(y.value_counts())

(3324, 308)
risk_label
2    1448
1    1195
0     681
Name: count, dtype: int64


# Model Training

**Step 1.** Train–Test Split

**Purpose :** To evaluate the model on unseen data.

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


##### Q. Why stratify=y?
--->
* Keeps class imbalance consistent in train & test
* Very important for risk problems

In [18]:
# Save feature names used during training
feature_columns = X_train.columns.tolist()

print("Total feature columns:", len(feature_columns))

Total feature columns: 308


**️Step 2.** Initialize the Final Model (Logistic Regression)

**Purpose :** Create the final model using production-safe settings.

In [19]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    n_jobs=-1
)


**Why these parameters:**

* max_iter=1000 → ensures convergence

* class_weight='balanced' → handles imbalance

* n_jobs=-1 → faster training

**️Step 3.** Train the Model

**Purpose :** Learn patterns from the training data.

In [20]:
model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, n_jobs=-1)

##### Now my model is sucessfully trained

In [21]:
print(model.classes_)

[0 1 2]


**Step 4.**  Make Predictions on Test Data

**Purpose :** Test how well the model generalizes.

In [22]:
y_pred = model.predict(X_test)

In [23]:
y_pred

array([1, 2, 1, 2, 0, 0, 2, 0, 1, 1, 2, 1, 2, 2, 1, 2, 2, 2, 2, 1, 2, 2,
       2, 1, 0, 1, 0, 0, 0, 1, 2, 2, 0, 2, 2, 2, 1, 1, 2, 2, 0, 2, 0, 2,
       2, 1, 2, 1, 2, 2, 1, 0, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, 1, 2, 0,
       0, 1, 0, 1, 1, 1, 1, 0, 2, 1, 1, 1, 2, 2, 1, 2, 2, 2, 0, 2, 2, 2,
       1, 1, 0, 1, 1, 2, 0, 2, 0, 2, 1, 1, 1, 2, 0, 2, 2, 0, 2, 1, 2, 2,
       2, 2, 1, 1, 2, 2, 2, 2, 2, 2, 0, 1, 2, 1, 1, 0, 2, 1, 1, 2, 0, 0,
       1, 1, 1, 0, 2, 0, 2, 2, 0, 0, 1, 2, 1, 1, 2, 2, 2, 0, 1, 2, 2, 2,
       2, 1, 1, 0, 1, 1, 2, 0, 2, 0, 2, 1, 0, 1, 1, 0, 2, 2, 2, 2, 0, 1,
       1, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 2, 1, 2, 1, 1, 2, 2, 2, 1, 2,
       0, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 1, 0, 0, 1, 0, 0, 0, 1, 2, 1, 0,
       1, 1, 2, 0, 2, 1, 2, 2, 2, 1, 1, 2, 1, 2, 1, 1, 0, 1, 1, 2, 1, 2,
       2, 2, 2, 2, 0, 2, 1, 2, 1, 2, 1, 0, 2, 1, 1, 1, 2, 1, 0, 2, 1, 2,
       0, 1, 2, 1, 1, 1, 0, 0, 1, 1, 1, 2, 2, 1, 2, 2, 1, 1, 0, 0, 2, 2,
       0, 1, 0, 2, 1, 0, 0, 2, 2, 1, 0, 2, 2, 0, 0,

In [24]:
sample_text = "Multiple unauthorized transactions detected and bank failed to respond"

cleaned = clean_text(sample_text)

features = create_features(
    text=cleaned,
    product="Credit Card",
    issue="Fraud"
)

import pandas as pd

df_sample = pd.DataFrame([features])
df_sample = df_sample.reindex(columns=feature_columns, fill_value=0)

print("Prediction:", model.predict(df_sample))
print("Probabilities:", model.predict_proba(df_sample))
print("Classes:", model.classes_)

Prediction: [0]
Probabilities: [[1.00000000e+00 2.20118567e-25 9.24936765e-45]]
Classes: [0 1 2]


**Step 5.** Evaluate Model Performance

**Purpose :** Measure quality using business-relevant metrics.

In [25]:
from sklearn.metrics import classification_report, f1_score

print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

F1 Score: 0.9458817952450916

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       136
           1       0.92      0.93      0.93       239
           2       0.95      0.93      0.94       290

    accuracy                           0.95       665
   macro avg       0.95      0.95      0.95       665
weighted avg       0.95      0.95      0.95       665



#####  Q. Why F1?
-->
* Balances precision & recall
* Better than accuracy for imbalanced data

**Step 6.** Confusion Matrix 

**Purpose :** Understand false positives & false negatives.

In [26]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred)

array([[135,   1,   0],
       [  2, 223,  14],
       [  0,  19, 271]], dtype=int64)

##### I use confusion matrix to analyze misclassification impact in risk detection.

**Step 7.** Save the Trained Model

**Purpose :** Use the model later in deployment (FastAPI).

In [27]:
import joblib

model_path = r"C:\Users\hp\Desktop\Fintech_Complaint_Analysis\models\logistic_regression_model.pkl"
joblib.dump(model, model_path)

print("Model saved successfully")


Model saved successfully


**Step 8.**  Save Feature Names 

**Purpose :** Ensure feature alignment during inference.

In [28]:
feature_cols_path = r"C:\Users\hp\Desktop\Fintech_Complaint_Analysis\models\feature_columns.pkl"
joblib.dump(X.columns.tolist(), feature_cols_path)

print("Feature columns saved")

Feature columns saved


In [29]:
print("Model training pipeline completed successfully")

Model training pipeline completed successfully
